# End-to-end V1 — Ingest → Bronze → Silver V1 → Gold V1 → Modeling

**Run all** in Google Colab. This is the **V1-only** workflow: it does not clean or read Silver V2 or Gold V2.

Stages: clone/install `.[data,modeling]` → mount Google Drive → verify/ingest a pinned 50K Bronze sample → verify/build Silver V1 → verify/build Gold V1 → sequential TF-IDF + Logistic Regression and TF-IDF + LinearSVC training → compare **validation Macro-F1**. Diagnostics are optional and disabled by default.

**Safety:** Data Lake snapshots and model artifacts are immutable. Existing completed artifacts are verified then skipped; partial/mismatched outputs raise errors. The held-out Gold test is **not used for model selection or evaluation**. Gold checksum verification reads all split files solely for integrity.

**Drive prerequisite:** Share the lake folder with the active Google account and add a shortcut under My Drive named `vietnamese-multilabel-topic-classification`, or set `DATA_LAKE_ROOT` to a real path inside the mounted Drive. The repository contains code, **not data**.

Use `VTC_GIT_REF` in the Colab environment to preview a feature branch. By default the notebook clones `main`. Run stages sequentially on CPU; GPU is optional and does not accelerate these sklearn baselines.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
BRANCH = os.environ.get("VTC_GIT_REF", "main")
PROJECT_DIR = Path("/content/vmm_colab_project")
if (PROJECT_DIR / ".git").is_dir():
    status = subprocess.check_output(
        ["git", "-C", str(PROJECT_DIR), "status", "--porcelain"], text=True
    )
    if status.strip():
        raise RuntimeError("Colab checkout is dirty; do not discard local modifications")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError("Project path exists but does not contain a Git repository")
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[data,modeling]"],
    cwd=PROJECT_DIR,
    check=True,
)
SRC_DIR = (PROJECT_DIR / "src").resolve()
sys.path[:] = [p for p in sys.path if p != str(SRC_DIR)]
sys.path.insert(0, str(SRC_DIR))

# A Colab kernel can preserve imports across git checkout and editable reinstall.
import importlib

prefix = "vietnamese_topic_classification"
for name in sorted(
    [key for key in sys.modules if key == prefix or key.startswith(prefix + ".")],
    key=len, reverse=True,
):
    del sys.modules[name]
importlib.invalidate_caches()

from vietnamese_topic_classification.config import get_domain_labels, load_domain_config
from vietnamese_topic_classification.data.bronze import BronzeReader
from vietnamese_topic_classification.data.colab_lake import resolve_mounted_lake
from vietnamese_topic_classification.data.gold import GoldConfig, GoldPipeline, GoldReader
from vietnamese_topic_classification.data.silver import SilverReader
from vietnamese_topic_classification.data.silver.v1 import SilverPipeline

revision = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], text=True,
).strip()
print("Repo:", BRANCH, "@", revision)


In [ ]:
from google.colab import drive

MOUNT = "/content/vtsnlp_drive"
drive.mount(MOUNT)
DATA_LAKE_ROOT = resolve_mounted_lake(MOUNT)
print("Shared Data Lake:", DATA_LAKE_ROOT)


In [ ]:
# V1 only: pin the same source and hyperparameters across reruns.
DATA_CONFIG = load_domain_config()
SAMPLE_SIZE = 50_000
SEED = 42
SHUFFLE_BUFFER = 10_000
MAX_FEATURES = 100_000
MIN_DF = 2
RUN_BALANCED = False  # Optional third baseline; two default baselines run sequentially.
RUN_DIAGNOSTICS = False  # Optional costly refits/plots, not required for the V1 run.

SOURCE_REVISION = DATA_CONFIG["dataset"]["revision"]
STAMP = f"{SOURCE_REVISION[:10]}_seed{SEED}_buf{SHUFFLE_BUFFER}_n{SAMPLE_SIZE}"
MANIFEST_NAME = f"source_manifest_{STAMP}.json"
BRONZE_PATH = DATA_LAKE_ROOT / "01_bronze" / f"vtsnlp_raw_sample_{STAMP}.jsonl"
BRONZE_MANIFEST = DATA_LAKE_ROOT / "00_source_metadata" / MANIFEST_NAME
BRONZE_REPORT = DATA_LAKE_ROOT / "04_reports" / f"bronze_ingestion_{STAMP}.json"
OUTPUT_ROOT = DATA_LAKE_ROOT / "experiments" / "tfidf_baselines_v1"
print("V1 source:", DATA_CONFIG["dataset"]["id"], SOURCE_REVISION)
print("Experiments:", OUTPUT_ROOT)


In [ ]:
# One-time ingestion from pinned VTSNLP dataset. Existing data are verified,
# not re-downloaded or overwritten.
bronze_existing = [p for p in (BRONZE_PATH, BRONZE_MANIFEST, BRONZE_REPORT) if p.exists()]
if bronze_existing:
    if len(bronze_existing) != 3:
        raise FileExistsError(f"Partial Bronze publication: {bronze_existing}")
    bronze_reader = BronzeReader(DATA_LAKE_ROOT, MANIFEST_NAME)
    bronze_verified = bronze_reader.verify()
    if bronze_verified["records"] != SAMPLE_SIZE:
        raise AssertionError("Bronze sample size does not match configuration")
    print("BRONZE VERIFIED — SKIPPED INGEST:", bronze_verified)
else:
    command = [
        sys.executable, str(PROJECT_DIR / "scripts/01_ingest_bronze.py"),
        "--output-root", str(DATA_LAKE_ROOT),
        "--sample-size", str(SAMPLE_SIZE),
        "--seed", str(SEED),
        "--shuffle-buffer", str(SHUFFLE_BUFFER),
    ]
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
    bronze_reader = BronzeReader(DATA_LAKE_ROOT, MANIFEST_NAME)
    bronze_verified = bronze_reader.verify()
    print("BRONZE INGESTED AND VERIFIED:", bronze_verified)


In [ ]:
# V1 layout: 02_silver/v1 and 04_reports/silver/v1.
silver_v1_pipeline = SilverPipeline(bronze_reader)
silver_v1_paths = silver_v1_pipeline.planned_paths()
v1_existing = [p for p in silver_v1_paths.values() if p.exists()]
v1_legacy = [
    DATA_LAKE_ROOT / "02_silver" / silver_v1_paths["silver"].name,
    DATA_LAKE_ROOT / "02_silver" / silver_v1_paths["rejected"].name,
    DATA_LAKE_ROOT / "04_reports" / silver_v1_paths["report"].name,
]
if any(p.exists() for p in v1_legacy):
    raise FileExistsError("Legacy V1 snapshot found outside v1/; inspect/migrate before publishing")
if v1_existing and len(v1_existing) != len(silver_v1_paths):
    raise FileExistsError(f"Partial Silver V1 publication: {v1_existing}")
if not v1_existing:
    silver_v1_pipeline.run()

v1_reader = SilverReader(DATA_LAKE_ROOT, silver_v1_paths["report"].name)
v1_verified = v1_reader.verify()
if v1_reader.report.get("source_sha256") != bronze_verified["sha256"]:
    raise AssertionError("Silver V1 lineage differs from verified Bronze")
print("SILVER V1 VERIFIED:", v1_verified)


In [ ]:
# Gold V1 is always derived from verified Silver V1; never mix in Silver V2.
gold_pipeline = GoldPipeline(v1_reader, config=GoldConfig(seed=SEED))
gold_paths = gold_pipeline.planned_paths(v1_verified["sha256"])
existing_gold = [p for p in gold_paths.values() if p.exists()]
stale_gold = [p for p in gold_paths.values() if p.with_name(p.name + ".uploading").exists()]
if stale_gold:
    raise FileExistsError(f"Stale Gold .uploading files: {stale_gold}")
if existing_gold:
    if len(existing_gold) != len(gold_paths):
        raise FileExistsError(f"Partial Gold V1 publication: {existing_gold}")
    gold_reader = GoldReader(DATA_LAKE_ROOT, gold_paths["manifest"].name)
    gold_verified = gold_reader.verify()
    print("GOLD V1 VERIFIED — SKIPPED BUILD:", gold_verified)
else:
    gold_pipeline.run()
    gold_reader = GoldReader(DATA_LAKE_ROOT, gold_paths["manifest"].name)
    gold_verified = gold_reader.verify()
    print("GOLD V1 PUBLISHED AND VERIFIED:", gold_verified)

if gold_reader.manifest["source"]["silver_sha256"] != v1_verified["sha256"]:
    raise AssertionError("Gold V1 source SHA256 differs from Silver V1")
if gold_reader.manifest["pipeline_version"] != "v1":
    raise AssertionError("Expected Gold pipeline V1")
GOLD_MANIFEST_NAME = gold_paths["manifest"].name
if gold_verified["split_rows"]["train"] < 2 or gold_verified["split_rows"]["validation"] < 1:
    raise ValueError("Gold train/validation too small for training")
print("Pinned Gold V1 manifest:", GOLD_MANIFEST_NAME)


In [ ]:
import hashlib
import json
import os

# SEED / feature parameters / RUN_BALANCED were pinned in the parameters cell.

# Hai baseline mặc định: LogisticRegression và LinearSVC, cùng TF-IDF.
EXPERIMENT_SPECS = [
    {"slug": "tfidf_logreg", "classifier": "logistic_regression", "class_weight": "none"},
    {"slug": "tfidf_linear_svc", "classifier": "linear_svc", "class_weight": "none"},
]
if RUN_BALANCED:
    EXPERIMENT_SPECS.append(
        {"slug": "tfidf_logreg_balanced",
         "classifier": "logistic_regression", "class_weight": "balanced"}
    )

if MAX_FEATURES < 1 or MIN_DF < 1:
    raise ValueError("MAX_FEATURES và MIN_DF phải dương.")

manifest_sha256 = hashlib.sha256(reader.manifest_path.read_bytes()).hexdigest()
for spec in EXPERIMENT_SPECS:
    spec["name"] = f"{spec['slug']}_v1_{manifest_sha256[:10]}"
    spec["dir"] = OUTPUT_ROOT / spec["name"]

def verify_existing_experiment(spec):
    """Chỉ skip khi model + reports hoàn chỉnh, cùng Gold source và hyperparams."""
    directory = spec["dir"]
    expected_files = ("model.joblib", "config.json", "metadata.json", "validation_metrics.json")
    missing = [name for name in expected_files if not (directory / name).is_file()]
    if missing:
        raise FileExistsError(f"Experiment tồn tại nhưng thiếu {missing}: {directory}")
    config = json.loads((directory / "config.json").read_text(encoding="utf-8"))
    metadata = json.loads((directory / "metadata.json").read_text(encoding="utf-8"))
    report = json.loads((directory / "validation_metrics.json").read_text(encoding="utf-8"))
    expected_weight = None if spec["class_weight"] == "none" else "balanced"
    if (
        config.get("gold_manifest_name") != GOLD_MANIFEST_NAME
        or config.get("seed") != SEED
        or config.get("classifier", {}).get("kind") != spec["classifier"]
        or config.get("classifier", {}).get("params", {}).get("class_weight") != expected_weight
        or config.get("features", {}).get("kind") != "tfidf"
        or config.get("features", {}).get("params", {}).get("max_features") != MAX_FEATURES
        or config.get("features", {}).get("params", {}).get("min_df") != MIN_DF
        or config.get("features", {}).get("params", {}).get("ngram_range") != [1, 2]
        or metadata.get("lineage", {}).get("gold_manifest_sha256") != manifest_sha256
        or not isinstance(report.get("macro_f1"), (int, float))
    ):
        raise ValueError(f"Artifact không khớp cấu hình/Gold hiện tại: {directory}")
    return report

# Preflight toàn bộ trước khi bắt đầu để không tự ghi đè artifacts cũ.
existing_reports = {}
for spec in EXPERIMENT_SPECS:
    if spec["dir"].exists():
        existing_reports[spec["name"]] = verify_existing_experiment(spec)
        print("VERIFIED EXISTING — SKIP:", spec["dir"])
    else:
        print("READY TO TRAIN:", spec["name"])
print("Tổng số baseline:", len(EXPERIMENT_SPECS))


In [ ]:
# Gọi training CLI gốc, tránh duplicate sklearn fit/metric logic trong notebook.
# Chạy TUẦN TỰ để hạn chế RAM trên Colab CPU.
env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(
    part for part in [str(SRC_DIR), env.get("PYTHONPATH", "")] if part
)

for spec in EXPERIMENT_SPECS:
    if spec["name"] in existing_reports:
        print("SKIPPED (verified):", spec["name"])
        continue
    command = [
        sys.executable, str(PROJECT_DIR / "scripts/06_train_baseline.py"),
        "--manifest-name", GOLD_MANIFEST_NAME,
        "--data-lake-root", str(DATA_LAKE_ROOT),
        "--output-root", str(OUTPUT_ROOT),
        "--name", spec["name"],
        "--classifier", spec["classifier"],
        "--class-weight", spec["class_weight"],
        "--max-features", str(MAX_FEATURES),
        "--min-df", str(MIN_DF),
        "--seed", str(SEED),
    ]
    print("TRAINING:", spec["name"], "(", spec["classifier"], ")")
    subprocess.run(command, cwd=str(PROJECT_DIR), env=env, check=True)
    existing_reports[spec["name"]] = verify_existing_experiment(spec)
    print("SAVED:", spec["dir"])
print("ALL BASELINE EXPERIMENTS READY")


In [ ]:
# Validation ONLY — chưa sử dụng test để chọn mô hình.
from IPython.display import Markdown, display

rows = []
for spec in EXPERIMENT_SPECS:
    metrics = verify_existing_experiment(spec)
    if len(metrics.get("per_class_f1", {})) != 25:
        raise AssertionError(f"Thiếu per-class F1 (25 nhãn): {spec['name']}")
    if len(metrics.get("confusion_matrix", [])) != 25:
        raise AssertionError(f"Confusion matrix không đủ 25 hàng: {spec['name']}")
    rows.append({
        "name": spec["name"],
        "classifier": spec["classifier"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
        "accuracy": metrics["accuracy"],
        "artifact_dir": str(spec["dir"]),
    })

rows.sort(key=lambda row: (-row["macro_f1"], row["name"]))
headers = "| Rank | Model | Macro-F1 | Weighted-F1 | Accuracy |"
divider = "|---:|---|---:|---:|---:|"
lines = [
    f"| {idx} | {row['name']} | {row['macro_f1']:.4f} "
    f"| {row['weighted_f1']:.4f} | {row['accuracy']:.4f} |"
    for idx, row in enumerate(rows, 1)
]
display(Markdown("\n".join([headers, divider, *lines])))
print("Best on VALIDATION Macro-F1:", rows[0]["name"])
print("Selected artifact:", rows[0]["artifact_dir"])
print("Test set CHƯA được đánh giá.")


In [ ]:
# Optional diagnostics; leave RUN_DIAGNOSTICS=False to avoid extra model refits.
if RUN_DIAGNOSTICS:
    for spec in EXPERIMENT_SPECS:
        command = [
            sys.executable, str(PROJECT_DIR / "scripts/08_plot_diagnostics.py"),
            "--data-lake-root", str(DATA_LAKE_ROOT),
            "--artifact-dir", str(spec["dir"]),
        ]
        subprocess.run(command, cwd=str(PROJECT_DIR), env=env, check=True)
        chart_dir = spec["dir"] / "diagnostics_v1"
        if not (chart_dir / "diagnostics.json").is_file():
            raise FileNotFoundError(chart_dir / "diagnostics.json")
        print("Diagnostics ready:", chart_dir)
else:
    print("Diagnostics skipped. Enable RUN_DIAGNOSTICS to create V1 charts.")


## Outputs and next step

- Bronze: `01_bronze/` plus `00_source_metadata/`; Silver V1: `02_silver/v1/` and `04_reports/silver/v1/`.
- Gold V1: `03_gold/gold_{train,validation,test}_v1_*.jsonl`, `03_gold/manifests/gold_manifest_v1_*.json` and `04_reports/gold_quality_v1_*.json`.
- Trained baselines: `experiments/tfidf_baselines_v1/<experiment_name>/` with model, config, metadata and validation metrics. Compare on **Macro-F1**; the two models share the same Gold V1 manifest.
- Test is held out until one model/configuration is finalized. `scripts/07_evaluate_model.py` is **not run here**.
- Source VTSNLP domain labels are automatically assigned; they are not independently verified human ground truth.

To inspect an individual stage, use the separate notebooks in `notebooks/v1/`. Silver V2 experiments are isolated in `notebooks/v2/`.
